# 1.0 Generating Protein Simulation Paramters



First we need to fix any structural issues with the protein coordinates and add the hydrogen atoms.  This will oriduce the `3LZT_out.cif`. 

In [33]:
import time
from Pras_Server.RunType import InitRunType
from Pras_Server.PDBID import  _82_pdbs, _494_pdbs

startTime = time.time()

fixing = InitRunType( rotamer="", mutation="", pdb_faspr="", keep_ligand="", chain_no="", 
              addh=False, ss=False, raman=False, ofname=False, pdbid=["3LZT.cif"], his_p=False)

fixing.ProcessWithoutDefaultUsingPDBID()

fixed 3LZT.cif


Then we can run martinize2 to produce a coarse-grained model for lysozyme using a Go model structural bias.

In [39]:
!martinize2 -f '3LZT_out.cif' -ff 'martini3001' -o '3lzt.top' -x '3lzt.pdb' -dssp -go

    INFO - general - _cell information missing from .cif file. Will write default dimensions
    INFO - step - Guessing the bonds.
    INFO - general - 1 molecules after guessing bonds
    INFO - step - Repairing the graph.
    INFO - general - Applying modification N-ter to residue A-LYS1
    INFO - general - Applying modification + to residue A-LYS1
    INFO - general - Applying modification C-ter to residue A-LEU129
    INFO - general - Applying modification + to residue A-LEU129
    INFO - step - Dealing with modifications.
    INFO - general - Identified the modifications ['N-ter'] on residues ['LYS1', 'LYS1', 'LYS1', 'LYS1']
    INFO - general - Identified the modifications ['C-ter'] on residues ['LEU129', 'LEU129', 'LEU129']
    INFO - step - Read input.
    INFO - step - Generating Go model contact map.
    INFO - general - Calculating Go contacts for 129 residues (1001 atoms)
    INFO - general - Contact map complete: 1198 Go contacts identified
    INFO - step - Creating the 

Before we proceed it is recommended to center the protein in a box.

In [40]:
! gmx editconf -f 3lzt.pdb -o 3lzt_center.pdb -box 10. 10. 10. -center 5. 5. 5

                :-) GROMACS - gmx editconf, 2024.2-Homebrew (-:

Executable:   /opt/homebrew/bin/../Cellar/gromacs/2024.2/bin/gmx
Data prefix:  /opt/homebrew/bin/../Cellar/gromacs/2024.2
Working dir:  /Users/fabian/ResearchData/PEGylation_new/PEGylation/tutorial_data
Command line:
  gmx editconf -f 3lzt.pdb -o 3lzt_center.pdb -box 10. 10. 10. -center 5. 5. 5

Note that major changes are planned in future for editconf, to improve usability and utility.
Read 433 atoms
No velocities found
    system size :  2.727  3.565  4.291 (nm)
    center      : -0.118  1.440  2.418 (nm)
    box vectors :  0.000  0.000  0.000 (nm)
    box angles  :   0.00   0.00   0.00 (degrees)
    box volume  :   0.00               (nm^3)
    shift       :  5.118  3.560  2.582 (nm)
new center      :  5.000  5.000  5.000 (nm)
new box vectors : 10.000 10.000 10.000 (nm)
new box angles  :  90.00  90.00  90.00 (degrees)
new box volume  :1000.00               (nm^3)

GROMACS reminds you: "It'll Cure Your Asthma Too !" (F

# 2 Generating PEGylated Protein Parameters

In [43]:
from pathlib import Path
from vermouth.forcefield import ForceField
from polyply import gen_params
from polyply.src.meta_molecule import MetaMolecule

ff = ForceField("test")

# step 1: load protein residue graph from ITP file
itp_file="molecule.itp" 
protein = MetaMolecule.from_itp(force_field=ff, 
                                itp_file="molecule.itp", 
                                mol_name="molecule")

# step 2: load PEG residue graph from CGsmiles string
cgsmiles_str="{[#APA][#EO]|10[#OHter]}"
polymer = MetaMolecule.from_cgsmiles_str(force_field=ff,
                                         cgsmiles_str=cgsmiles_str, 
                                         mol_name="PEO")

# step 3: add PEG to protein and connect CYS114 to first PEG unit
protein.merge_meta_mol(polymer, connects=[(33, 0)])

gen_params(inpath=[Path("molecule.itp"), Path("PEGylation.martini3.ff")],
           lib=["martini3"],
           meta_molecule=protein,
           name="lysoPEG",
           outpath=Path("lysoPEG.itp"))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 659.31it/s]


# 3 Structure generation and simulation of PEGylated Proteins

In [44]:
toplines="""#include "martini_v3.0.0.itp"
#include "go_atomtypes.itp"
#include "go_nbparams.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "lysoPEG.itp"

[ system ]
Title of the system

[ molecules ]
lysoPEG 1
W 10000
NA 100
CL 108"""
with open("lysoPEG.top", "w") as _file:
    _file.write(toplines)

In [45]:
from polyply import gen_coords
import gromacs as gmx
import os

# step 1: generate the missing PEGylation coordinates and solvate
gen_coords(toppath=Path("lysoPEG.top"), 
           density=1000, 
           outpath=Path("start.gro"), 
           coordpath=Path("3lzt_center.pdb"),
           name="lysoPEG")

A density is provided via the command line, but the starting coordinates define a box.Will try to pack all molecules in the box provided with starting coordinates.
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10209/10209 [00:01<00:00, 6415.78it/s]


In [23]:
os.mkdir("simulation")
os.chdir("simulation")
# step 2: equilibriate the system and start production simulation
prev_stage="start"
for stage in ["mini", "nvt", "prod"]:
    gmx.grompp(f=f"./cg_mdps/{stage}.mdp",
               c=f"{prev_stage}.gro", 
               p="lysoPEG.top",
               o=f"{stage}.tpr")
    gmx.mdrun(deffnm=stage, 
              v=False, 
              s=f"{stage}.tpr")

                 :-) GROMACS - gmx grompp, 2024.2-Homebrew (-:

Executable:   /opt/homebrew/bin/../Cellar/gromacs/2024.2/bin/gmx
Data prefix:  /opt/homebrew/bin/../Cellar/gromacs/2024.2
Working dir:  /Users/fabian/ResearchData/PEGylation_new/PEGylation/tutorial_data
Command line:
  gmx grompp -f ./cg_mdps/mini.mdp -c start.gro -p lysoPEG.top -o mini.tpr

Ignoring obsolete mdp entry 'ns_type'

NOTE 1 [file ./cg_mdps/mini.mdp]:
  verlet-buffer-pressure-tolerance is ignored when verlet-buffer-tolerance
  < 0


NOTE 2 [file ./cg_mdps/mini.mdp]:
  Setting tcoupl from 'V-rescale' to 'no'. Temperature coupling does not
  apply to steep.


NOTE 3 [file ./cg_mdps/mini.mdp]:
  Setting pcoupl from 'Parrinello-Rahman' to 'no'. Pressure coupling does
  not apply to steep.

Number of degrees of freedom in T-Coupling group System is 32095.00
The integrator does not provide a ensemble temperature, there is no system ensemble temperature

There were 3 NOTEs

Back Off! I just backed up mini.tpr to ./#mi

Setting the LD random seed to 2080193373

Generated 117840 of the 473851 non-bonded parameter combinations

Excluding 1 bonded neighbours molecule type 'lysoPEG'

Excluding 1 bonded neighbours molecule type 'W'

Excluding 1 bonded neighbours molecule type 'NA'

Excluding 1 bonded neighbours molecule type 'CL'

Cleaning up constraints and constant bonded interactions with virtual sites
Analysing residue names:
There are:   258    Protein residues
There are: 10012      Other residues
There are:   408        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...

This run will generate roughly 4 Mb of data



Energy minimization reached the maximum number of steps before the forces
reached the requested precision Fmax < 10.

writing lowest energy coordinates.

Back Off! I just backed up mini.gro to ./#mini.gro.1#

Steepest Descents did not converge to Fmax < 10 in 6001 steps.
Potential Energy  = -3.1956312e+05
Maximum force     =  4.2592412e+03 on atom 148
Norm of force     =  6.4722449e+01

GROMACS reminds you: "The plural of regex is regrets" (Steve McCarthy (on Twitter))

                 :-) GROMACS - gmx grompp, 2024.2-Homebrew (-:

Executable:   /opt/homebrew/bin/../Cellar/gromacs/2024.2/bin/gmx
Data prefix:  /opt/homebrew/bin/../Cellar/gromacs/2024.2
Working dir:  /Users/fabian/ResearchData/PEGylation_new/PEGylation/tutorial_data
Command line:
  gmx grompp -f ./cg_mdps/nvt.mdp -c start.gro -p lysoPEG.top -o nvt.tpr


NOTE 1 [file ./cg_mdps/nvt.mdp]:
  verlet-buffer-pressure-tolerance is ignored when verlet-buffer-tolerance
  < 0

Number of degrees of freedom in T-Coupling group Syst

Setting the LD random seed to 2147413724

Generated 117840 of the 473851 non-bonded parameter combinations

Excluding 1 bonded neighbours molecule type 'lysoPEG'

Excluding 1 bonded neighbours molecule type 'W'

Excluding 1 bonded neighbours molecule type 'NA'

Excluding 1 bonded neighbours molecule type 'CL'

Velocities were taken from a Maxwell distribution at 310 K

Cleaning up constraints and constant bonded interactions with virtual sites
Analysing residue names:
There are:   258    Protein residues
There are: 10012      Other residues
There are:   408        Ion residues
Analysing Protein...
Analysing residues not classified as Protein/DNA/RNA/Water and splitting into groups...

This run will generate roughly 35 Mb of data


Wrote pdb files with previous and current coordinates

Step 3, time 0.03 (ps)  LINCS WARNING
relative constraint deviation after LINCS:
rms 775700217856.000000, max 9688452169728.000000 (between atoms 76 and 77)
bonds that rotated more than 50 degrees:
 atom 1 atom 2  angle  previous, current, constraint length
     20     22  161.1    3.7127   7.6504      0.3100
     22     24   70.3    4.9138   4.7161      0.3100
     76     78   70.2  6565.8408 23997.8008      0.3100
    206    208   82.0    1.2811   9.0896      0.3300
    212    214   96.9    3.3748  11.9870      0.3100
    214    216  106.6    1.1395   3.4555      0.3100
    218    220   84.8    1.8787   3.1659      0.3100
    225    228   97.7    2.0327   6.3894      0.3100
    228    230  110.8    3.4077   2.4909      0.3100
    239    240  138.6    0.5794   1.5390      0.3300
    240    242  170.7    1.6072   1.4408      0.3100
    242    244  114.2    2.9908   5.3012      0.3100
    244    246  128.4    2.9463   3.9411      0.

GromacsError: [Errno 1] Gromacs tool failed
Command invocation: gmx mdrun -deffnm nvt -nov -s nvt.tpr

In [33]:
import nglview as nv
view = nv.show_file("3lzt.pdb")
view

NGLWidget()

In [4]:
idx_nodes ={0:4,1:2}
sorted_mol_nodes = sorted(idx_nodes, key=idx_nodes.get)


In [5]:
sorted_mol_nodes

[1, 0]